# Legal AI Agent — Colab GPU Embedding Pipeline

**Mục tiêu:** Embed 122,000+ văn bản pháp luật VN trên GPU Colab, lưu kết quả về Google Drive.

| Bước | Thời gian ước tính |
|------|-------------------|
| Download data (~3.5 GB) | 35–50 phút |
| Ingest + Embed (T4 GPU) | **3–5 giờ** |
| Build BM25 | 10–15 phút |
| **Tổng** | **~4–6 giờ** |

> ⚠️ **Quan trọng:** Colab free giới hạn 12h/session. Notebook đã hỗ trợ **resume** — nếu bị ngắt, chạy lại từ Cell 8 với flag `--skip-existing`.

## Cell 1 — Kiểm tra GPU & Disk

In [ ]:
import torch, os, time

# GPU check
cuda_ok = torch.cuda.is_available()
if cuda_ok:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU  : {gpu_name} ({vram_gb:.1f} GB VRAM)')
    print(f'  batch_size sẽ dùng: 64 (T4) hoặc 128 (A100)')
    BATCH_SIZE = 128 if 'A100' in gpu_name else 64
else:
    print('⚠  KHÔNG CÓ GPU! → Runtime → Change runtime type → T4 GPU')
    BATCH_SIZE = 16

# Disk check
import shutil
total, used, free = shutil.disk_usage('/content')
print(f'\n💾 Disk: {free//1e9:.0f} GB free / {total//1e9:.0f} GB total')
if free < 15e9:
    print('⚠  Cần ít nhất 15 GB free. Xóa bớt file hoặc dùng Runtime mới.')
else:
    print('✓ Disk space OK')

print(f'\nBATCH_SIZE = {BATCH_SIZE}')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/LegalAI_vectorstore'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT}/chroma',  exist_ok=True)
print(f'✓ Drive mounted')
print(f'  Output: {DRIVE_OUTPUT}')
!ls -lh /content/drive/MyDrive/LegalAI_vectorstore/ 2>/dev/null || echo '  (thư mục trống — lần đầu chạy)'

## Cell 3 — Clone repo & cài dependencies

In [ ]:
os.chdir('/content')

# Clone hoặc pull nếu đã có
if os.path.exists('/content/ProjectGenAI_2/.git'):
    print('Repo đã có, pulling latest...')
    os.chdir('/content/ProjectGenAI_2')
    !git pull origin main
else:
    !git clone https://github.com/HoangNhatTR/ProjectGenAI_2.git
    os.chdir('/content/ProjectGenAI_2')

print('\nInstalling dependencies...')
!pip install -q -r requirements.txt
print('✓ Done')

## Cell 4 — Cấu hình .env

> ✏️ Điền API keys của bạn vào đây trước khi chạy.

In [ ]:
# ── Điền API keys của bạn ──────────────────────────────────────────────────
ROUTER9_API_KEY = ""   # NineRouter key — lấy tại router9 dashboard
KIEAI_API_KEY   = ""   # Kie AI key — lấy tại: https://kie.ai/api-key
GEMINI_API_KEY  = ""   # Gemini key (tùy chọn)
# ──────────────────────────────────────────────────────────────────────────

VSTORE_DIR = '/content/drive/MyDrive/LegalAI_vectorstore/chroma'

env_content = f"""# Auto-generated for Colab — {time.strftime('%Y-%m-%d %H:%M')}

# LLM
LLM_PROVIDER=kieai
LLM_MODEL=deepseek-chat
ROUTER9_API_KEY={ROUTER9_API_KEY}
ROUTER9_BASE_URL=http://localhost:20128/v1
ROUTER9_MODEL=cc/claude-haiku-4-5-20251001
KIEAI_API_KEY={KIEAI_API_KEY}
KIEAI_BASE_URL=https://kieai.erweima.ai/api/v1
GEMINI_API_KEY={GEMINI_API_KEY}

# Embedding (GPU tự động dùng CUDA)
EMBEDDING_MODEL=BAAI/bge-m3

# Vectorstore → lưu thẳng lên Drive
VECTORSTORE_DIR={VSTORE_DIR}
COLLECTION_NAME=legal_docs

# Chunking
CHUNK_SIZE=600
CHUNK_OVERLAP=80
TOP_K=5

# Parent-Child (Điều → Khoản → Điểm)
USE_PARENT_CHILD=true
PARENT_STORE_PATH=/content/ProjectGenAI_2/data/processed/parent_store.db

# HyDE — tắt trong lúc ingest
USE_HYDE=false
"""

with open('/content/ProjectGenAI_2/.env', 'w') as f:
    f.write(env_content)

os.makedirs('/content/ProjectGenAI_2/data/processed', exist_ok=True)
os.environ['VECTORSTORE_DIR'] = VSTORE_DIR

print('✓ .env created')
!grep -v 'API_KEY' /content/ProjectGenAI_2/.env | grep -v '^$'

## Cell 5 — Download data từ HuggingFace (~35–50 phút)

> Lần đầu: tải 88k+ VB từ thuvienphapluat.vn. Lần sau: resume tự động.

In [ ]:
os.chdir('/content/ProjectGenAI_2')
!df -h /content

print('\n⏳ Downloading data (~88k văn bản, ~3.5 GB)...')
t0 = time.time()
# --resume: tự bỏ qua các VB đã tải trước đó
!python -m scripts.load_hf_dataset --resume
print(f'\n⏱ Tải xong sau {(time.time()-t0)/60:.0f} phút')

## Cell 6 — Kiểm tra data

In [ ]:
from pathlib import Path

raw_dir = Path('/content/ProjectGenAI_2/data/raw')
grand_total = 0
grand_mb    = 0

for folder in sorted(raw_dir.iterdir()):
    if not folder.is_dir():
        continue
    txts = list(folder.rglob('*.txt'))
    if not txts:
        continue
    mb = sum(f.stat().st_size for f in txts) / 1e6
    print(f'  {folder.name:25s}: {len(txts):7,} files | {mb:6.0f} MB')
    grand_total += len(txts)
    grand_mb    += mb

root_txts = list(raw_dir.glob('*.txt'))
print(f'  {"(root)".ljust(25)}: {len(root_txts):7,} files')
grand_total += len(root_txts)

print(f'\n  TỔNG : {grand_total:,} files | {grand_mb:.0f} MB ({grand_mb/1024:.1f} GB)')

if grand_total < 100_000:
    print('⚠  Ít hơn 100k files — data chưa đủ. Chạy lại Cell 5.')
else:
    print('✓ Data đầy đủ, sẵn sàng ingest')

!df -h /content

## Cell 7 — Patch Embedder cho GPU

In [ ]:
import sys
sys.path.insert(0, '/content/ProjectGenAI_2')

from dotenv import load_dotenv
load_dotenv('/content/ProjectGenAI_2/.env')

import torch
device     = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = BATCH_SIZE  # từ Cell 1

# Patch Embedder để dùng GPU + batch_size tối ưu
from src import embedding as emb_module

def _gpu_init(self, model_name, device=None, batch_size=16, max_seq_length=1024):
    self.model_name    = model_name
    self.device        = 'cuda' if torch.cuda.is_available() else 'cpu'
    self.batch_size    = BATCH_SIZE
    self.max_seq_length = max_seq_length
    self.top_p         = 1.0   # compat với generator attr
    self._model        = None
    print(f'  Embedder: device={self.device}, batch_size={self.batch_size}')

emb_module.Embedder.__init__ = _gpu_init
print(f'✓ Embedder patched → GPU batch_size={BATCH_SIZE}')

# Pre-load model để kiểm tra
print('Pre-loading bge-m3 (~1 GB download)...')
from src import config
from src.embedding import Embedder
emb = Embedder(config.EMBEDDING_MODEL)
test_emb = emb.encode(['test'])
print(f'✓ Model loaded, embedding dim = {len(test_emb[0])}')

## Cell 8 — Ingest (Embedding → Vectorstore)

> ⚡ **Bước chính — mất 3–5 giờ trên T4 GPU.**
>
> Nếu Colab bị ngắt giữa chừng: chạy lại Cell 7 → Cell 8 với `--skip-existing` thay vì `--reset`.
>
> **Lần đầu:** dùng `--reset` để xóa vectorstore cũ và tạo mới với chunking mới (parent-child + point-level).

In [ ]:
os.chdir('/content/ProjectGenAI_2')
os.environ['VECTORSTORE_DIR'] = '/content/drive/MyDrive/LegalAI_vectorstore/chroma'
os.makedirs(os.environ['VECTORSTORE_DIR'], exist_ok=True)

# ── Chọn mode ─────────────────────────────────────────────────────────────
# Lần đầu chạy:    INGEST_FLAG = '--reset'
# Resume sau timeout: INGEST_FLAG = '--skip-existing'
INGEST_FLAG = '--reset'  # <── ĐỔI THÀNH '--skip-existing' khi resume
# ──────────────────────────────────────────────────────────────────────────

print(f'🚀 Ingest mode: {INGEST_FLAG}')
print(f'   Vectorstore → {os.environ["VECTORSTORE_DIR"]}')
print(f'   Parent-Child → data/processed/parent_store.db')
print('   (Chunk theo Điều → Khoản → Điểm a/b/c + parent context)')
print()

t0 = time.time()
!python -m scripts.ingest {INGEST_FLAG}
elapsed = (time.time() - t0) / 3600
print(f'\n✓ Ingest xong sau {elapsed:.1f} giờ')

## Cell 9 — Lưu Parent Store về Drive

In [ ]:
import shutil

src_db = '/content/ProjectGenAI_2/data/processed/parent_store.db'
dst_db = '/content/drive/MyDrive/LegalAI_vectorstore/parent_store.db'

if os.path.exists(src_db):
    shutil.copy2(src_db, dst_db)
    mb = os.path.getsize(dst_db) / 1e6
    print(f'✓ parent_store.db → Drive ({mb:.0f} MB)')
else:
    print('⚠ parent_store.db không tìm thấy!')
    print('  Kiểm tra: USE_PARENT_CHILD=true trong .env')

## Cell 10 — Build BM25 index

In [ ]:
os.chdir('/content/ProjectGenAI_2')
print('⏳ Building BM25 index từ vectorstore...')
t0 = time.time()
!python -m scripts.build_bm25
print(f'⏱ BM25 done: {(time.time()-t0)/60:.0f} phút')

# Copy BM25 về Drive
bm25_src = Path('/content/ProjectGenAI_2/data/bm25')
bm25_dst = '/content/drive/MyDrive/LegalAI_vectorstore/bm25'
if bm25_src.exists():
    shutil.copytree(str(bm25_src), bm25_dst, dirs_exist_ok=True)
    mb = sum(f.stat().st_size for f in Path(bm25_dst).rglob('*') if f.is_file()) / 1e6
    print(f'✓ BM25 index → Drive ({mb:.0f} MB)')
else:
    print('⚠ BM25 chưa được tạo')

## Cell 11 — Verify kết quả

In [ ]:
from src import config
from src.vectorstore import VectorStore
from src.parent_store import ParentStore

print('=== VECTORSTORE ===')
store = VectorStore(config.VECTORSTORE_DIR, config.COLLECTION_NAME)
n_chunks = store.count()
print(f'  Chunks : {n_chunks:,}')

# Sample chunk để kiểm tra parent_id và point
sample = list(store.iter_all_chunks(batch_size=200))[:200]
n_with_point  = sum(1 for c in sample if c.point)
n_with_parent = sum(1 for c in sample if c.parent_id)
print(f'  Có point (Điểm a/b/c): {n_with_point}/200 mẫu {"✓" if n_with_point > 0 else "⚠ = 0 (kiểm tra lại)"}')
print(f'  Có parent_id         : {n_with_parent}/200 mẫu {"✓" if n_with_parent > 0 else "⚠ = 0 (kiểm tra lại)"}')

print('\n=== PARENT STORE ===')
ps_path = '/content/drive/MyDrive/LegalAI_vectorstore/parent_store.db'
if os.path.exists(ps_path):
    ps = ParentStore(Path(ps_path))
    print(f'  Parents: {ps.count():,}')
else:
    print('  ⚠ Không tìm thấy')

print('\n=== BM25 ===')
bm25_path = Path('/content/drive/MyDrive/LegalAI_vectorstore/bm25')
if bm25_path.exists():
    mb = sum(f.stat().st_size for f in bm25_path.rglob('*') if f.is_file()) / 1e6
    print(f'  Size: {mb:.0f} MB ✓')
else:
    print('  ⚠ Không tìm thấy')

print('\n=== DRIVE FILES ===')
!ls -lh /content/drive/MyDrive/LegalAI_vectorstore/

## Cell 12 — Test RAG query

In [ ]:
from src.embedding import Embedder
from src.retriever import Retriever
from src.bm25_index import BM25Index
from src.parent_store import ParentStore

embedder  = Embedder(config.EMBEDDING_MODEL)
store_obj = VectorStore(config.VECTORSTORE_DIR, config.COLLECTION_NAME)

# Load BM25
bm25_index_path = Path('/content/ProjectGenAI_2/data/bm25/index.json')
bm25 = BM25Index(bm25_index_path) if bm25_index_path.exists() else None

# Load ParentStore
ps_path = Path('/content/ProjectGenAI_2/data/processed/parent_store.db')
ps = ParentStore(ps_path) if ps_path.exists() else None

retriever = Retriever(
    embedder=embedder, store=store_obj,
    bm25=bm25, parent_store=ps
)

queries = [
    'Mức phạt vượt đèn đỏ xe máy là bao nhiêu?',
    'Điều kiện được cấp sổ đỏ lần đầu',
    'Sa thải trái luật bồi thường bao nhiêu tháng lương?',
]

for query in queries:
    results = retriever.retrieve(query, top_k=3, use_kg=False, use_parent_expansion=(ps is not None))
    print(f'\n🔍 "{query}"')
    for i, r in enumerate(results, 1):
        loc  = ' > '.join(filter(None, [r.chunk.article, r.chunk.clause, r.chunk.point]))
        src  = (r.chunk.metadata.doc_number or r.chunk.metadata.title or '?')[:35]
        pid  = '✓pid' if r.chunk.parent_id else ''
        print(f'  [{i}] {r.score:.3f} | {src} | {loc or "(no loc)"} {pid}')
        print(f'       {r.chunk.text[:100].strip()}...')

## Hướng dẫn deploy lên server

Sau khi notebook chạy xong, Google Drive sẽ có:
```
LegalAI_vectorstore/
├── chroma/           ← Vectorstore (~vài GB)
├── parent_store.db   ← Parent chunks (~1-2 GB)
└── bm25/             ← BM25 index (~30-50 MB)
```

### Trên server:
```bash
git clone https://github.com/HoangNhatTR/ProjectGenAI_2.git && cd ProjectGenAI_2
pip install -r requirements.txt

# Tải vectorstore từ Drive (dùng rclone)
rclone copy drive:LegalAI_vectorstore ./data/
# Hoặc download thủ công từ Google Drive

# Cấu hình .env
cp .env.example .env
# Sửa: VECTORSTORE_DIR=./data/chroma
#      PARENT_STORE_PATH=./data/parent_store.db
#      KIEAI_API_KEY=...

# Chạy API
python api.py
# → http://localhost:8000
```